# 多言語トポニム Harmonizer 実験ノート

**DEMO FILE**: このノートブックはデモ用に作成されたものです。

## 概要
このノートブックでは、アマゾン流域の地名（トポニム）を分析し、水に関連する地名を抽出・分類するプロセスを実験します。

### 主な処理ステップ
1. データ収集（BNGB API、OpenStreetMap）
2. 前処理（正規化、アクセント除去）
3. 埋め込み生成（多言語Sentence Transformer）
4. クラスタリング（k-NN、水関連語彙との類似度）
5. 確率マップ生成（`Pwater(x)` ラスター）

In [ ]:
# 必要なライブラリのインポート
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import rasterio
from rasterio.features import rasterize
import unidecode
import re
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

# プロジェクトのルートディレクトリをパスに追加
sys.path.append(os.path.abspath('../'))

# 自作パッケージのインポート
# from src.tamagawa_to_z.harmonizer import preprocess, embed, cluster

## 1. サンプルデータの読み込み

実際のプロジェクトでは、BNGB APIやOpenStreetMapから地名データを取得します。ここではデモ用のサンプルデータを使用します。

In [ ]:
# デモ用サンプルデータ
sample_data = {
    'name': [
        'Igarapé do Paxiúba', 'Rio Baixio', 'Lago Grande', 'Paraná do Ramos',
        'Igarapé Açu', 'Rio Amazonas', 'Cachoeira Alta', 'Serra do Moa',
        'Cidade Nova', 'Floresta Nacional', 'Igarapé do Macaco', 'Monte Alegre'
    ],
    'lat': [
        -3.1042, -3.2156, -2.4278, -2.7361,
        -3.0125, -3.1489, -2.9872, -3.5214,
        -3.0781, -3.4269, -2.8753, -2.0014
    ],
    'lon': [
        -59.9731, -59.8642, -59.2458, -58.2467,
        -59.1235, -59.9864, -58.7531, -60.1245,
        -59.5678, -59.3214, -58.9876, -59.1234
    ],
    'feature_type': [
        'stream', 'river', 'lake', 'channel',
        'stream', 'river', 'waterfall', 'mountain',
        'settlement', 'forest', 'stream', 'settlement'
    ],
    'language': [
        'portuguese', 'portuguese', 'portuguese', 'portuguese',
        'portuguese', 'portuguese', 'portuguese', 'portuguese',
        'portuguese', 'portuguese', 'portuguese', 'portuguese'
    ]
}

# DataFrameに変換
df = pd.DataFrame(sample_data)

# GeoPandasに変換
gdf = gpd.GeoDataFrame(
    df, 
    geometry=[Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])],
    crs="EPSG:4326"
)

gdf.head()

## 2. 前処理（正規化、アクセント除去）

地名を正規化し、アクセント記号を除去して比較しやすくします。

In [ ]:
# 正規化関数
def normalize_toponym(name):
    """地名を正規化する関数
    
    1. 小文字化
    2. アクセント除去
    3. 特殊文字を空白に置換
    4. 前後の空白を削除
    """
    # アクセント除去と小文字化
    normalized = unidecode.unidecode(name.lower())
    # 英数字とスペース、ハイフン以外を空白に置換
    normalized = re.sub(r'[^a-z0-9\s-]', ' ', normalized)
    # 余分な空白を削除
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

# 正規化を適用
gdf['normalized_name'] = gdf['name'].apply(normalize_toponym)

# 結果を確認
pd.DataFrame({
    'original': gdf['name'],
    'normalized': gdf['normalized_name']
})

## 3. 埋め込み生成（多言語Sentence Transformer）

正規化した地名をSentence Transformerを使って埋め込みベクトルに変換します。

In [ ]:
# Sentence Transformerモデルのロード
# 注：実際の実行時には時間がかかります
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-v2')

# 埋め込みの生成
embeddings = model.encode(gdf['normalized_name'].tolist())

# 埋め込みの次元を確認
print(f"埋め込みの形状: {embeddings.shape}")

# 最初の埋め込みベクトルの一部を表示
print(f"最初の埋め込みベクトル（最初の10次元）: {embeddings[0][:10]}")

## 4. 水関連語彙との類似度計算

水に関連する語彙のシードリストを用意し、各地名との類似度を計算します。

In [ ]:
# 水関連語彙のシードリスト（ポルトガル語）
water_seeds = [
    "rio", "igarape", "lago", "parana", "cachoeira", 
    "corrego", "lagoa", "canal", "baia", "represa"
]

# シード語彙の正規化
normalized_seeds = [normalize_toponym(seed) for seed in water_seeds]

# シード語彙の埋め込み
seed_embeddings = model.encode(normalized_seeds)

# 最近傍探索器の初期化
nn = NearestNeighbors(n_neighbors=3, metric='cosine')
nn.fit(seed_embeddings)

# 各地名埋め込みに対して最近傍のシード語彙を探索
distances, indices = nn.kneighbors(embeddings)

# 類似度スコアの計算（コサイン距離を類似度に変換）
similarity_scores = 1 - distances.mean(axis=1)

# 結果をDataFrameに追加
gdf['water_similarity'] = similarity_scores

# 水関連度が高い順にソート
gdf_sorted = gdf.sort_values('water_similarity', ascending=False)

# 結果を表示
pd.DataFrame({
    'name': gdf_sorted['name'],
    'normalized': gdf_sorted['normalized_name'],
    'feature_type': gdf_sorted['feature_type'],
    'water_similarity': gdf_sorted['water_similarity'].round(3)
})

## 5. 水関連確率マップの生成

各地名の水関連度をもとに、空間上の確率マップ（`Pwater(x)` ラスター）を生成します。

In [ ]:
# 地理座標系から投影座標系に変換（面積計算のため）
gdf_projected = gdf.to_crs("EPSG:3857")

# ラスターの範囲とセルサイズを定義
xmin, ymin, xmax, ymax = gdf_projected.total_bounds
cell_size = 1000  # 1kmのセルサイズ
width = int((xmax - xmin) / cell_size)
height = int((ymax - ymin) / cell_size)

# 各ポイントから影響範囲（バッファ）を作成
gdf_projected['buffer'] = gdf_projected.buffer(5000)  # 5kmのバッファ
gdf_projected['weight'] = gdf_projected['water_similarity']

# ラスター化の準備
shapes = [(geom, value) for geom, value in zip(gdf_projected['buffer'], gdf_projected['weight'])]
transform = rasterio.transform.from_bounds(xmin, ymin, xmax, ymax, width, height)

# ラスター化（各セルは重なるバッファの最大値を取る）
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    all_touched=True,
    merge_alg=rasterio.enums.MergeAlg.max
)

# 結果の可視化
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(raster, cmap='Blues', extent=[xmin, xmax, ymin, ymax])
gdf_projected.plot(ax=ax, color='red', markersize=20)
for idx, row in gdf_projected.iterrows():
    ax.annotate(row['name'], xy=(row.geometry.x, row.geometry.y), xytext=(3, 3), 
                textcoords="offset points", fontsize=8)
plt.colorbar(im, ax=ax, label='Pwater(x) - 水関連確率')
plt.title('水関連地名の確率マップ')
plt.tight_layout()
plt.show()

## 6. 結論と次のステップ

このノートブックでは、多言語トポニムの解析と水関連確率マップの生成の基本的なワークフローを示しました。

### 次のステップ
1. より大規模なデータセットでの検証
2. 言語検出と多言語対応の強化
3. LLMを用いた語源分析の追加
4. 水理データ同化システムとの連携（`02_hydro_da.ipynb`で実施）

### 参考文献
- BNGB (IBGE API): https://ibge.gov.br
- OpenStreetMap / Overpass API: https://wiki.openstreetmap.org
- Sentence Transformers: https://huggingface.co/sentence-transformers